In [ ]:
#| default_exp features.fsd

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class FSDEvaluator(FeatureEvaluator):
    """Extracts fragment size distribution frequencies from 10bp to 400bp."""
    
    name = "FSD"
    source_file = ".FSD.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
                
            cols = set(df.columns)
            
            # FSD files typically have columns representing sizes, e.g. "50", "100", or "size", "frequency"
            # Assuming a row-wise distribution curve where rows = sizes or there's a matrix structure
            # To be safe, if we get raw frequencies over sizes:
            if "size" in cols and "frequency" in cols:
                # Capture exact frequencies for primary sizes 
                target_sizes = [166, 143, 332]
                for s in target_sizes:
                    val = df.loc[df["size"] == s, "frequency"]
                    if not val.empty:
                        extracted[f"fsd_freq_{s}bp"] = float(val.iloc[0])
                        
                # Summary metrics
                extracted["fsd_modal_size"] = float(df.loc[df["frequency"].idxmax(), "size"])
            
            return extracted
        except Exception as e:
            log.warning("fsd_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = FSDEvaluator()
df_test = pd.DataFrame({"size": [143, 166], "frequency": [0.05, 0.12]})
res = evaluator.extract(df_test)
assert res["fsd_freq_166bp"] == 0.12
assert res["fsd_modal_size"] == 166
